# Historical Pricing get_data_async with Asyncio.Gather Performance (30 RICs)

Importing requires libraries

In [1]:
import os
import time
import asyncio
from IPython.display import Markdown, display
import lseg.data as ld
from lseg.data import session
from lseg.data._errors import LDError
from lseg.data.content import historical_pricing
from lseg.data.content.historical_pricing import Adjustments, Intervals
from dotenv import load_dotenv

Loading Platform session (Data Platform) credential from `.env` file.

In [2]:
# Load environment variables from .env file
load_dotenv(dotenv_path='.env')
# Retrieve Platform Session credentials from environment variables
app_key = os.getenv('LSEG_API_KEY')
username = os.getenv('LSEG_MACHINE_ID')
password = os.getenv('LSEG_PASSWORD')

Define required variables.

In [3]:
# -- Top 30 S&P 500 companies ----------------------------------------------------
INSTRUMENTS = {
    "NVDA.O":  "NVIDIA",
    "AAPL.O":  "Apple",
    "MSFT.O":  "Microsoft",
    "AMZN.O":  "Amazon",
    "GOOG.O":  "Alphabet",
    "AVGO.O":  "Broadcom",
    "META.O":  "Meta",
    "ORCL.N":  "Oracle",
    "IBM.N":   "IBM",
    "PLTR.O":  "Palantir",
    "NFLX.O":  "Netflix",
    "TSLA.O":  "Tesla",
    "CRM.N":   "Salesforce",
    "AMD.O":   "AMD",
    "INTC.O":  "Intel",
    "ARM.O":   "Arm Holdings",
    "TXN.O": "Texas Instruments",
    "CSCO.O":  "Cisco Systems",
    "WMT.O":   "Walmart",
    "LLY.N":   "Eli Lilly and Company",
    "JPM.N":   "JPMorgan Chase & Co.",
    "XOM.N":   "Exxon Mobil Corporation",
    "V.N":     "Visa Inc.",
    "JNJ.N":   "Johnson & Johnson",
    "MU.O":    "Micron Technology, Inc.",
    "MA.N":    "Mastercard Incorporated",
    "COST.O":  "Costco Wholesale Corporation",
    "CVX.N":   "Chevron Corporation",
    "BAC.N":   "Bank of America Corporation",
    "CAT.N":   "Caterpillar Inc.",
}

# ── Date range ─────────────────────────────────────────────────────────────────
START = "2025-11-01T00:00:00Z"
END   = "2026-02-28T23:59:59Z"

# ── Event correction filters ───────────────────────────────────────────────────
EVENT_ADJUSTMENTS = [Adjustments.EXCHANGE_CORRECTION,
                    Adjustments.MANUAL_CORRECTION,
                    Adjustments.CCH,
                    Adjustments.CRE,
                    Adjustments.RPO,
                    Adjustments.RTS]

# ── Field lists ─────────────────────────────────────────────────────────────────
INTERDAY_FIELDS = ["BID", "ASK", "OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "NUM_MOVES", "TRNOVR_UNS"]
MAX_CONCURRENT_REQUESTS = 10  # Maximum number of concurrent requests to the API

Initialize and open the session.

In [5]:
# Create a platform session with provided credentials for authentication
ld_session = session.platform.Definition(
         app_key=app_key,
         grant=session.platform.GrantPassword(
             username=username,
             password=password
         ),
         signon_control=True
).get_session()

# Set this session as the default for all subsequent Data Library calls
session.set_default(ld_session)

# Open the connection to the LSEG Data Platform
ld_session.open()

<OpenState.Opened: 'Opened'>

Define `display_response` that handles `asyncio.gather` with `return_exceptions=True`.

In [6]:
def display_response(data, format="df"):
    """Display the result of each async API call.

    For each response:
    - Prints the exception message if the task raised a Python error.
    - Warns if the response has no closure label (unexpected type).
    - Renders the DataFrame on a successful HTTP response.
    - Prints the HTTP status code on a failed (4xx/5xx) response.
    """
    for api_response in data:
        # Task raised a Python exception (e.g. network error, timeout)
        if isinstance(api_response, Exception):
            print(f"\nTask failed with exception: {api_response}")
            continue

        # Guard against unexpected response types
        if not hasattr(api_response, 'closure'):
            print(f"\nUnexpected response type: {type(api_response)}")
            continue

        print(f"\nResponse received for: {api_response.closure}")

        if api_response.is_success:
            if format == "df":
                display(api_response.data.df)
            elif format == "json":
                display(api_response.data.raw)
        else:
            # HTTP-level failure (4xx / 5xx from the platform)
            print(f"Request failed — HTTP status: {api_response.http_status}")

Requests data for 30 instruments asynchronously while measuring total execution time. Concurrency is managed with `asyncio.Semaphore(10)` to throttle request throughput and reduce the likelihood of rate-limit responses.



In [7]:
rics = list(INSTRUMENTS.keys())

# Convert the company dictionary to a list once.
# Example item: ("NVDA.O", "NVIDIA")
list_of_rics_companies = list(INSTRUMENTS.items())

throttle_limit = MAX_CONCURRENT_REQUESTS
semaphore = asyncio.Semaphore(throttle_limit)  # Number of simultaneous tasks to run

async def fetch_with_throttle(ric, company):
    """Fetch interday data for a given RIC and company with concurrency limit."""
    async with semaphore:
        return await historical_pricing.summaries.Definition(
            universe=ric,
            fields=INTERDAY_FIELDS,
            count=5,
            interval=Intervals.DAILY,
        ).get_data_async(closure=company)

display(Markdown("**Start the wall-clock timer...**"))
start_time = time.perf_counter()

try:
    historical_data = await asyncio.gather(  # pylint: disable=await-outside-async
        *[
            fetch_with_throttle(ric, company)
            for ric, company in list_of_rics_companies
        ],
        return_exceptions=True,
    )

except* LDError as errors:
    for error in errors.exceptions:
        print(error)
finally:
    elapsed = time.perf_counter() - start_time
    elapsed_string = f"**Requesting Historical-Pricing Interday Historical Summaries (Daily) for ({len(rics)}) RICs with concurrency limit of {throttle_limit} ends in {elapsed:0.2f} seconds.**"
    display(Markdown(elapsed_string))

**Start the wall-clock timer...**

**Requesting Historical-Pricing Interday Historical Summaries (Daily) for (30) RICs with concurrency limit of 10 ends in 12.71 seconds.**

The code uses `response.data.raw` statement to display the raw output data in JSON format. If you instead use `response.data.df` statement to retrieve the data as a DataFrame, please note that additional time is required to convert the JSON data into a DataFrame.

In [8]:
try:
    display(Markdown(f"**({len(rics)}) Companies Historical Price Interday Historical Summaries (Daily) via Iteration**"))
    display_response(historical_data, format="json")
except Exception as errors:
    print(f"An exception occurred: {errors}")

**(30) Companies Historical Price Interday Historical Summaries (Daily) via Iteration**


Response received for: NVIDIA


{'universe': {'ric': 'NVDA.O'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-06-25',
   195.74,
   200.8,
   192.13,
   200.08,
   195.72,
   195.76,
   29368587068,
   2767503],
  ['2026-06-24',
   199,
   201.67,
   196.58,
   200.12,
   198.95,
   198.98,
   30250233652,
   2539558],
  ['2026-06-2


Response received for: Apple


{'universe': {'ric': 'AAPL.O'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-06-25',
   275.15,
   288.8,
   273.75,
   287.4,
   275.06,
   275.16,
   29757167342,
   1767825],
  ['2026-06-24',
   293.08,
   299.7,
   292.94,
   295.355,
   293.07,
   293.08,
   15651500922,
   927710],
  ['2026-06-

...

{'universe': {'ric': 'CVX.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-06-25',
   172.24,
   172.715,
   169.32,
   169.82,
   172.24,
   172.28,
   355866837,
   13397],
  ['2026-06-24',
   171.45,
   173.39,
   170.86,
   173.12,
   171.55,
   171.58,
   526625855,
   16114],
  ['2026-06-23',



Response received for: Bank of America Corporation


{'universe': {'ric': 'BAC.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-06-25',
   58.19,
   59.19,
   57.935,
   58.06,
   58.19,
   58.2,
   561616524,
   31134],
  ['2026-06-24', 57.73, 58.33, 57.39, 58.21, 57.75, 57.77, 614967310, 28924],
  ['2026-06-23', 57.91, 58.005, 57.2, 57.52, 57.92, 57


Response received for: Caterpillar Inc.


{'universe': {'ric': 'CAT.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-06-25',
   1057.01,
   1057.07,
   1022.46,
   1024.29,
   1056.7,
   1057,
   1105302549,
   19385],
  ['2026-06-24',
   994.45,
   1004.51,
   970.68,
   981.61,
   994.41,
   994.73,
   849411511,
   13452],
  ['2026-06-23

Closing the session.

In [9]:
# Close the session to release resources; in a long-running application, consider keeping the session open and reusing it for subsequent API calls instead.
ld_session.close()
ld.close_session()